# nano-deepseek-v4 — Tutorial 01
## A CPU-first executable tour

The default path in this notebook is deliberately small and offline. It:

1. runs the native tiny hybrid model on CPU,
2. trains one step and saves a tokenizer-bound v2 bundle,
3. reloads that bundle for text generation,
4. runs the fixed core conformance profile, and
5. inspects the Flash-0731 architecture and its packaged metadata receipt without allocating the model or downloading checkpoint weights.

The 159.6 GB original Flash snapshot and full model materialization are kept in an explicit opt-in appendix.

## 0. Imports and environment

Install the current source checkout before starting. The default cells need only the core package and PyTorch.

In [ ]:
import json
import tempfile
from importlib.resources import files
from pathlib import Path

import torch

import nano_deepseek_v4 as ndv4
from nano_deepseek_v4.architecture import inspect_architecture
from nano_deepseek_v4.conformance import run_conformance
from nano_deepseek_v4.generate_text import run_text_generation
from nano_deepseek_v4.train_text import run_tiny_text_training

print('nano_deepseek_v4 version:', ndv4.__version__)
print('torch version:           ', torch.__version__)
print('cuda available:          ', torch.cuda.is_available())

## 1. Run the tiny native architecture

`DeepSeekV4Config()` is a roughly one-million-parameter model that still exercises sliding, CSA, and HCA attention, mHC residual streams, hash and learned MoE routing, and MTP.

In [ ]:
torch.manual_seed(0)
tiny_config = ndv4.DeepSeekV4Config()
tiny_model = ndv4.DeepSeekV4ForCausalLM(tiny_config).eval()
tiny_ids = torch.randint(0, tiny_config.vocab_size, (1, 16))

with torch.no_grad():
    tiny_output = tiny_model(tiny_ids, labels=tiny_ids)

tiny_parameter_count = sum(parameter.numel() for parameter in tiny_model.parameters())
assert tuple(tiny_output.logits.shape) == (1, 16, tiny_config.vocab_size)
assert torch.isfinite(tiny_output.loss)
assert tiny_output.mtp_loss is not None
print(f'parameters: {tiny_parameter_count:,}')
print('attention: ', ' -> '.join(tiny_config.layer_types))
print('logits:    ', tuple(tiny_output.logits.shape))

## 2. Train one step and round-trip a tokenizer-bound bundle

The training helper uses its built-in byte corpus, writes a format-v2 native bundle into a temporary workspace, reloads the model and tokenizer, and checks exact logits. The workspace remains available while this kernel is alive and is removed when the kernel exits.

In [ ]:
_tutorial_workspace = tempfile.TemporaryDirectory(prefix='nano-deepseek-v4-tutorial-')
tutorial_root = Path(_tutorial_workspace.name)
bundle_dir = tutorial_root / 'tiny-bundle'

training_result = run_tiny_text_training(
    steps=1,
    context_length=8,
    batch_size=1,
    eval_batches=1,
    learning_rate=1e-3,
    seed=7,
    device='cpu',
    max_new_tokens=1,
    save_directory=bundle_dir,
)

assert bundle_dir.is_dir()
assert training_result.bundle_format_version == 2
assert training_result.checkpoint_round_trip_match is True
assert training_result.tokenizer_round_trip_match is True
print('bundle:              ', bundle_dir)
print('trained tokens:      ', training_result.trained_tokens)
print('bundle round-trip:   ', training_result.checkpoint_round_trip_match)
print('tokenizer round-trip:', training_result.tokenizer_round_trip_match)

## 3. Generate from that exact bundle

Generation verifies the bundle and its checksum-bound byte tokenizer before loading it.

In [ ]:
generation_result = run_text_generation(
    bundle_dir,
    prompt='DeepSeek',
    max_new_tokens=1,
    device='cpu',
    seed=7,
)

assert generation_result.generated_new_tokens == 1
assert generation_result.bundle_manifest_sha256 == training_result.bundle_manifest_sha256
print('generated text:      ', repr(generation_result.generated_text))
print('continuation tokens:', generation_result.continuation_token_ids)

## 4. Run the core conformance profile

The core profile runs the tiny full/cache check, allocation-free Flash-0731 architecture consistency, and deterministic attention reachability. Network access is forbidden by its standard-library socket guard.

In [ ]:
conformance_report = run_conformance(profile='core')

assert conformance_report.passed
assert conformance_report.complete
assert conformance_report.network['policy'] == 'forbidden'
assert conformance_report.network['attempted'] is False
print('status: ', conformance_report.status)
print('checks: ', conformance_report.summary)
print('network:', conformance_report.network)

## 5. Inspect Flash-0731 without allocating it

`flash_0731()` describes the 43-layer backbone plus its three DSpark stages. `inspect_architecture` performs logical parameter accounting from the config alone. The packaged canonical receipt records a pinned metadata/header inspection; reading it here performs no network request and does not verify weight payload integrity.

In [ ]:
flash_0731_config = ndv4.DeepSeekV4Config.flash_0731()
flash_0731_architecture = inspect_architecture(
    flash_0731_config,
    source='DeepSeekV4Config.flash_0731()',
)

receipt_path = (
    files('nano_deepseek_v4')
    .joinpath('_receipts')
    .joinpath('DeepSeek-V4-Flash-0731-metadata.json')
)
flash_0731_metadata = json.loads(receipt_path.read_text(encoding='utf-8'))
dspark_metadata = flash_0731_metadata['dspark_namespace']

assert flash_0731_architecture.auxiliary_kind == 'dspark'
assert flash_0731_architecture.dspark_stage_count == 3
assert flash_0731_architecture.runtime_load_supported is False
assert dspark_metadata['runtime_load_supported'] is False
assert dspark_metadata['dtype_counts']['I8'] == 2304
print('logical parameters: ', flash_0731_architecture.total_parameter_count)
print('DSpark stages:      ', flash_0731_architecture.dspark_stage_count)
print('metadata revision:  ', flash_0731_metadata['source_revision'])
print('namespace tensors:  ', dspark_metadata['inspected_tensor_count'])
print('packed FP4 dtype:   ', 'signed I8 raw bytes')
print('runtime supported:  ', dspark_metadata['runtime_load_supported'])

To replay the pinned Flash-0731 metadata/header inspection, install the optional Hub dependency and run:

```bash
python -m pip install -e ".[official]"
nano-deepseek-v4 inspect \
  --hf-repo deepseek-ai/DeepSeek-V4-Flash-0731 \
  --revision 7872f01b1d1fe23eabc4c98b48bffcef5a386062 \
  --namespace mtp --json
nano-deepseek-v4 verify-flash-0731 --json
```

That live path fetches pinned config/index documents and selected shard-header ranges. It is not a full-snapshot or runtime-execution claim.

## Appendix — original Flash full snapshot (explicit opt-in)

The original `deepseek-ai/DeepSeek-V4-Flash` snapshot is about 159.6 GB across 46 safetensors shards. Its packed FP4 experts are stored with signed I8 headers; the loader treats signed or unsigned byte tensors as raw bit patterns before decoding E2M1 nibbles.

The three flags below are all `False` by default. Download, full metadata preflight, and materialization are separate choices. Materialization has a much higher peak-memory requirement than the checkpoint size and is not a serving path.

In [ ]:
FULL_FLASH_REPO = 'deepseek-ai/DeepSeek-V4-Flash'
FULL_FLASH_REVISION = 'fd53f944496234770ba80e15004f9b6d269a71f5'
FULL_FLASH_DIR = Path('./checkpoints/DeepSeek-V4-Flash')
DOWNLOAD_FULL_FLASH = False
RUN_FULL_FLASH_PREFLIGHT = False
MATERIALIZE_FULL_FLASH = False

if DOWNLOAD_FULL_FLASH:
    from huggingface_hub import snapshot_download

    snapshot_download(
        repo_id=FULL_FLASH_REPO,
        revision=FULL_FLASH_REVISION,
        local_dir=FULL_FLASH_DIR,
    )
else:
    print('Full Flash download skipped (159.6 GB).')

In [ ]:
full_flash_preflight = None
full_flash_coverage = None
if RUN_FULL_FLASH_PREFLIGHT:
    if not (FULL_FLASH_DIR / 'config.json').is_file():
        raise FileNotFoundError(
            'Set DOWNLOAD_FULL_FLASH=True or point FULL_FLASH_DIR at a complete snapshot.'
        )
    full_flash_preflight = ndv4.verify_deepseek_checkpoint_snapshot(FULL_FLASH_DIR)
    full_flash_coverage = ndv4.analyze_deepseek_official_index(
        FULL_FLASH_DIR / 'model.safetensors.index.json'
    )
    assert full_flash_preflight.is_complete
    print('indexed tensors:  ', full_flash_preflight.total_keys)
    print('recognized keys:  ', len(full_flash_coverage.recognized_keys))
    print('scale sidecars:   ', len(full_flash_coverage.scale_keys))
    print('unrecognized keys:', len(full_flash_coverage.unrecognized_keys))
else:
    print('Full Flash preflight skipped.')

In [ ]:
full_flash_load_report = None
if MATERIALIZE_FULL_FLASH:
    if not (FULL_FLASH_DIR / 'config.json').is_file():
        raise FileNotFoundError('FULL_FLASH_DIR does not contain config.json.')
    full_flash_config = ndv4.DeepSeekV4Config.from_official_json(
        FULL_FLASH_DIR / 'config.json'
    )
    torch.set_default_dtype(torch.bfloat16)
    full_flash_model = ndv4.DeepSeekV4ForCausalLM(full_flash_config)
    full_flash_load_report = ndv4.load_deepseek_official_checkpoint(
        full_flash_model,
        FULL_FLASH_DIR,
    )
    print('converted keys:', len(full_flash_load_report.conversion.converted_keys))
else:
    print('Full Flash materialization skipped.')

## Next steps

- Use `nano-deepseek-v4 --help` for the complete command surface.
- Read `docs/guides/train-and-generate.md` for durable bundles and longer experiments.
- Read `docs/guides/official-checkpoints.md` for the exact metadata, payload-integrity, and runtime claim boundaries.